# ANDES Dataset Generator for Dynamic State Estimation (DSE)

This notebook generates a synthetic DSE dataset inspired by the paper:

- J. Pei, P. Wang, J. Wang, and D. Shi, “Deep generative model-aided power system dynamic state estimation and reconstruction with unknown control inputs or data distributions,” arXiv:2501.02928, Jan. 2025, doi: 10.48550/arXiv.2501.02928. [Online]. Available: https://arxiv.org/abs/2501.02928

The paper uses the **ANDES python library**, which is work of:

- H. Cui, F. Li, and K. Tomsovic, “Hybrid symbolic-numeric framework for power system modeling and analysis,” IEEE Transactions on Power Systems, vol. 36, no. 2, pp. 1373–1384, Mar. 2021, doi: 10.1109/TPWRS.2020.3017019. [Online]. Available: https://ieeexplore.ieee.org/document/9169830

Core convention:

> **One sample = one simulated event trajectory**, not one CSV row.

Each trajectory is saved as:

```text
output_dir/
  split/
    event_type/
      seed_<seed>/
        scenario_name/
          config.json
          truth.csv
          meas_clean.csv
          meas_noisy.csv
          noise_meta.csv
```

Assumptions used here because the paragraph does not specify them:

| Item | Assumption |
|---|---:|
| Event occurrence time | 1.0 s |
| Simulation horizon | 10.0 s |
| Internal TDS step | 1/120 s |
| Exported input/target sampling | 1/60 s |
| Fault duration | 0.05–0.12 s |
| Fault impedance | rf 0.0–0.02 pu, xf 0.01–0.08 pu |
| Line trips | 1–2 lines, optional reclosure |
| Load changes | ±5–20% on 1–3 loads |
| Dataset split | train: faults/line trips/load changes; test: generator shedding + remaining events |

> Jump to **Section 2** if you want to change the config.

> Jump to **Sections 10, 11, 12** for the results and **EDA**.

## 1. Imports

In [1]:
%matplotlib inline

import json
from copy import deepcopy
from pathlib import Path

import andes
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

andes.config_logger(stream_level=30)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

## 2. Configuration

Start with `COUNTS_PILOT`. After verifying the generator and `manifest.csv`, switch to `COUNTS_PAPER_LIKE`.

In [2]:
CFG = {
    # Replace with the IEEE 39-bus case if available in your ANDES installation.
    # This default uses a built-in dynamic case to keep the notebook runnable.
    "case_file": andes.get_case("ieee14/ieee14_full.xlsx"),
    "output_dir": Path("output/ieee14_full_pilot_v2"),
    "base_seed": 2026,
    # Assumed simulation settings.
    "sim_tf": 10.0,
    "sim_tstep": 1 / 120,
    "sample_tstep": 1 / 60,
    "event_time": 1.0,
    # Paper-like GENROU targets: delta, omega, E'q, E'd.
    "truth_vars": {
        "gen_delta": ("GENROU", "delta"),
        "gen_omega": ("GENROU", "omega"),
        "gen_e1q": ("GENROU", "e1q"),
        "gen_e1d": ("GENROU", "e1d"),
    },
    # PMU-like measurements. Avoid adding target states here unless leakage is intentional.
    "meas_vars": {
        "bus_v": ("Bus", "v"),
        "bus_a": ("Bus", "a"),
    },
    "noise_std": {
        "bus_v": 0.002,
        "bus_a": 0.002,
    },
    "event_ranges": {
        "fault_duration": (0.05, 0.12),
        "fault_rf": (0.0, 0.02),
        "fault_xf": (0.01, 0.08),
        "line_reclose_probability": 0.70,
        "line_reclose_delay": (0.10, 0.50),
        "n_lines_per_event": (1, 2),
        "load_change_fraction_abs": (0.05, 0.20),
        "load_change_signs": (-1, 1),
        "n_loads_per_event": (1, 3),
        "n_generators_shed_per_event": (1, 1),
    },
    "continue_on_error": True,
    # TDS options.
    # Default: keep ANDES stability criteria active.
    # For generator shedding: disable the stability stop, because shedding can
    # produce loss-of-synchronism trajectories that are still useful as test data.
    "tds_options": {
        "default": {
            "criteria": 1,
            "ddelta_limit": 180,
            "shrinkt": 1,
            "max_iter": 15,
            "tol": 1e-4,
        },
        "event_overrides": {
            "generator_shedding": {
                "criteria": 0,
                # More conservative numerical settings for severe events.
                # If this becomes too slow, change back to 1 / 120.
                "tstep": 1 / 240,
                "max_iter": 30,
                "tol": 1e-5,
            }
        },
    },
}

CFG["output_dir"].mkdir(parents=True, exist_ok=True)

COUNTS_PILOT = {
    "train": {"fault": 5, "line_trip": 5, "load_change": 3},
    "test": {"generator_shedding": 10, "fault": 2, "line_trip": 2, "load_change": 2},
}

COUNTS_MEDIUM = {
    "train": {"fault": 50, "line_trip": 30, "load_change": 20},
    "test": {"generator_shedding": 50, "fault": 10, "line_trip": 10, "load_change": 10},
}

# 5000 trajectories total:
# 3000 train from faults, line trips, load changes.
# 1250 test generator-shedding trajectories.
# 750 remaining test trajectories from other event types.
COUNTS_ANDES_PAPER = {
    "train": {"fault": 1500, "line_trip": 1000, "load_change": 500},
    "test": {
        "generator_shedding": 1250,
        "fault": 250,
        "line_trip": 250,
        "load_change": 250,
    },
}

ACTIVE_COUNTS = COUNTS_PILOT

CFG

{'case_file': '/mnt/datadisk/conda/envs/juan_datasets_utilities/lib/python3.10/site-packages/andes/cases/ieee14/ieee14_full.xlsx',
 'output_dir': PosixPath('output/ieee14_full_pilot_v2'),
 'base_seed': 2026,
 'sim_tf': 10.0,
 'sim_tstep': 0.008333333333333333,
 'sample_tstep': 0.016666666666666666,
 'event_time': 1.0,
 'truth_vars': {'gen_delta': ('GENROU', 'delta'),
  'gen_omega': ('GENROU', 'omega'),
  'gen_e1q': ('GENROU', 'e1q'),
  'gen_e1d': ('GENROU', 'e1d')},
 'meas_vars': {'bus_v': ('Bus', 'v'), 'bus_a': ('Bus', 'a')},
 'noise_std': {'bus_v': 0.002, 'bus_a': 0.002},
 'event_ranges': {'fault_duration': (0.05, 0.12),
  'fault_rf': (0.0, 0.02),
  'fault_xf': (0.01, 0.08),
  'line_reclose_probability': 0.7,
  'line_reclose_delay': (0.1, 0.5),
  'n_lines_per_event': (1, 2),
  'load_change_fraction_abs': (0.05, 0.2),
  'load_change_signs': (-1, 1),
  'n_loads_per_event': (1, 3),
  'n_generators_shed_per_event': (1, 1)},
 'continue_on_error': True,
 'tds_options': {'default': {'criter

## 3. Inspect the selected case and requested variables

In [3]:
def inspect_case(cfg):
    ss = andes.load(cfg["case_file"], setup=False)

    print("ANDES version:", getattr(andes, "__version__", "unknown"))
    print("Case file:", cfg["case_file"])
    print("Has TDS.get_timeseries:", hasattr(ss.TDS, "get_timeseries"))
    print()

    for model_name in [
        "Bus",
        "Line",
        "GENROU",
        "GENCLS",
        "PQ",
        "Toggle",
        "Fault",
        "Alter",
    ]:
        if hasattr(ss, model_name):
            model = getattr(ss, model_name)
            idx = list(model.idx.v) if hasattr(model, "idx") else []
            print(
                f"{model_name}: n={len(idx)}, idx={idx[:10]}{' ...' if len(idx) > 10 else ''}"
            )
        else:
            print(f"{model_name}: not present")

    print("\nRequested variables:")
    all_specs = {}
    all_specs.update(cfg["truth_vars"])
    all_specs.update(cfg["meas_vars"])

    for label, (model_name, var_name) in all_specs.items():
        ok = hasattr(ss, model_name) and hasattr(getattr(ss, model_name), var_name)
        print(f"  {label}: {model_name}.{var_name} -> {ok}")
        if ok:
            var = getattr(getattr(ss, model_name), var_name)
            print(f"    class={type(var).__name__}, addresses={np.asarray(var.a)}")

    return ss


probe = inspect_case(CFG)

ANDES version: 1.10.1
Case file: /mnt/datadisk/conda/envs/juan_datasets_utilities/lib/python3.10/site-packages/andes/cases/ieee14/ieee14_full.xlsx
Has TDS.get_timeseries: False

Bus: n=14, idx=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10] ...
Line: n=20, idx=['Line_1', 'Line_2', 'Line_3', 'Line_4', 'Line_5', 'Line_6', 'Line_7', 'Line_8', 'Line_9', 'Line_10'] ...
GENROU: n=5, idx=['GENROU_1', 'GENROU_2', 'GENROU_3', 'GENROU_4', 'GENROU_5']
GENCLS: n=0, idx=[]
PQ: n=11, idx=['PQ_1', 'PQ_2', 'PQ_3', 'PQ_4', 'PQ_5', 'PQ_6', 'PQ_7', 'PQ_8', 'PQ_9', 'PQ_10'] ...
Toggle: n=0, idx=[]
Fault: n=0, idx=[]
Alter: n=0, idx=[]

Requested variables:
  gen_delta: GENROU.delta -> True
    class=State, addresses=[]
  gen_omega: GENROU.omega -> True
    class=State, addresses=[]
  gen_e1q: GENROU.e1q -> True
    class=State, addresses=[]
  gen_e1d: GENROU.e1d -> True
    class=State, addresses=[]
  bus_v: Bus.v -> True
    class=Algeb, addresses=[]
  bus_a: Bus.a -> True
    class=Algeb, addresses=[]


## 4. Extraction helpers

These support both newer ANDES versions with `TDS.get_timeseries()` and ANDES 1.10.1-style access through `ss.dae.ts`.

In [4]:
def resolve_var(ss, model_name, var_name):
    return getattr(getattr(ss, model_name), var_name)


def _var_class_name(var_obj):
    return type(var_obj).__name__.lower()


def _legacy_get_timeseries(ss, model_name, var_name):
    model = getattr(ss, model_name)
    var_obj = getattr(model, var_name)

    t = np.asarray(ss.dae.ts.t, dtype=float)
    addr = np.atleast_1d(np.asarray(var_obj.a, dtype=int))
    cls = _var_class_name(var_obj)

    if cls in ("state", "extstate"):
        hist = np.asarray(ss.dae.ts.x, dtype=float)
    elif cls in ("algeb", "extalgeb"):
        hist = np.asarray(ss.dae.ts.y, dtype=float)
    else:
        raise TypeError(
            f"Unsupported variable class {type(var_obj).__name__} for {model_name}.{var_name}"
        )

    if hist.ndim != 2:
        raise ValueError(
            f"Unexpected history shape for {model_name}.{var_name}: {hist.shape}"
        )

    if hist.shape[0] == len(t):
        hist2 = hist
    elif hist.shape[1] == len(t):
        hist2 = hist.T
    else:
        raise ValueError(
            f"Cannot align {model_name}.{var_name}: len(t)={len(t)}, hist={hist.shape}"
        )

    values = hist2[:, addr]
    cols = list(model.idx.v)
    if len(cols) != values.shape[1]:
        cols = cols[: values.shape[1]]

    df = pd.DataFrame(values, index=t, columns=cols)
    df.index.name = "t"
    return df


def get_var_timeseries(ss, model_name, var_name):
    var_obj = resolve_var(ss, model_name, var_name)
    if hasattr(ss.TDS, "get_timeseries"):
        return ss.TDS.get_timeseries(var_obj).copy()
    return _legacy_get_timeseries(ss, model_name, var_name)


def extract_frame(ss, spec_dict):
    frames = []
    for label, (model_name, var_name) in spec_dict.items():
        df = get_var_timeseries(ss, model_name, var_name)
        df.columns = [f"{label}__{col}" for col in df.columns]
        frames.append(df)

    out = pd.concat(frames, axis=1).sort_index()
    out.index.name = "t"
    return out


def resample_df(df, dt):
    t_old = df.index.to_numpy(dtype=float)
    t_new = np.arange(t_old[0], t_old[-1] + 1e-12, dt)

    out = pd.DataFrame(index=t_new)
    for col in df.columns:
        out[col] = np.interp(t_new, t_old, df[col].to_numpy(dtype=float))

    out.index.name = "t"
    return out

## 5. Event generation

The event types are: `fault`, `line_trip`, `generator_shedding`, and `load_change`.

`load_change` uses `Alter` on `PQ.Ppf` and `PQ.Qpf`, because dynamic PQ-load changes in ANDES are normally applied through these effective TDS quantities rather than `p0/q0`.

In [5]:
def get_indices(ss, model_name):
    if not hasattr(ss, model_name):
        return []
    model = getattr(ss, model_name)
    if not hasattr(model, "idx"):
        return []
    return list(model.idx.v)


def randint_range(rng, bounds):
    low, high = bounds
    return int(rng.integers(low, high + 1))


def sample_without_replacement(rng, items, n):
    """
    Sample up to n items without replacement.

    This version samples integer positions instead of sampling the objects
    directly, so it works for strings, integers, tuples, and dictionaries.
    """
    items = list(items)

    if len(items) == 0:
        return []

    n = min(n, len(items))

    selected_positions = rng.choice(len(items), size=n, replace=False)

    return [items[int(pos)] for pos in np.atleast_1d(selected_positions)]


def build_event_definition(ss_probe, cfg, split, event_type, event_idx, seed):
    rng = np.random.default_rng(seed)
    ranges = cfg["event_ranges"]
    t0 = cfg["event_time"]

    buses = get_indices(ss_probe, "Bus")
    lines = get_indices(ss_probe, "Line")
    genrou = get_indices(ss_probe, "GENROU")
    gencls = get_indices(ss_probe, "GENCLS")
    pq_loads = get_indices(ss_probe, "PQ")

    if event_type == "fault":
        duration = float(rng.uniform(*ranges["fault_duration"]))
        return {
            "type": "fault",
            "bus": int(rng.choice(buses)),
            "tf": t0,
            "tc": t0 + duration,
            "rf": float(rng.uniform(*ranges["fault_rf"])),
            "xf": float(rng.uniform(*ranges["fault_xf"])),
        }

    if event_type == "line_trip":
        n_lines = randint_range(rng, ranges["n_lines_per_event"])
        selected_lines = sample_without_replacement(rng, lines, n_lines)
        reclose = bool(rng.random() < ranges["line_reclose_probability"])
        delay = float(rng.uniform(*ranges["line_reclose_delay"])) if reclose else None
        return {
            "type": "line_trip",
            "lines": selected_lines,
            "trip_t": t0,
            "reclose_t": t0 + delay if delay is not None else None,
        }

    if event_type == "generator_shedding":
        available = [("GENROU", idx) for idx in genrou] + [
            ("GENCLS", idx) for idx in gencls
        ]

        # Avoid shedding generator 1 by default.
        # In many test systems, generator 1 is reference/slack-like or structurally important.
        available = [(model, idx) for model, idx in available if str(idx) != "1"]

        if len(available) == 0:
            raise ValueError(
                "No non-reference generators available for generator shedding."
            )

        n_gens = randint_range(rng, ranges["n_generators_shed_per_event"])
        selected = sample_without_replacement(rng, available, n_gens)

        return {
            "type": "generator_shedding",
            "generators": [{"model": model, "dev": dev} for model, dev in selected],
            # Paper-like assumption: event occurs at t = 1 s.
            "shed_t": t0,
        }

    if event_type == "load_change":
        n_loads = randint_range(rng, ranges["n_loads_per_event"])
        selected_loads = sample_without_replacement(rng, pq_loads, n_loads)
        changes = []

        for load_id in selected_loads:
            frac_abs = float(rng.uniform(*ranges["load_change_fraction_abs"]))
            sign = int(rng.choice(ranges["load_change_signs"]))
            multiplier = 1.0 + sign * frac_abs
            changes.append(
                {
                    "model": "PQ",
                    "dev": load_id,
                    "active_src": "Ppf",
                    "reactive_src": "Qpf",
                    "method": "*",
                    "amount": multiplier,
                }
            )

        return {"type": "load_change", "changes": changes, "change_t": t0}

    raise ValueError(f"Unsupported event type: {event_type}")


def build_scenario_table(cfg, counts):
    ss_probe = andes.load(cfg["case_file"], setup=False)
    rows = []
    global_i = 0

    for split, event_counts in counts.items():
        for event_type, n_events in event_counts.items():
            for k in range(n_events):
                seed = cfg["base_seed"] + global_i
                rows.append(
                    {
                        "sample_id": global_i,
                        "split": split,
                        "event_type": event_type,
                        "scenario_name": f"{event_type}_{k:05d}",
                        "seed": seed,
                        "event": build_event_definition(
                            ss_probe, cfg, split, event_type, k, seed
                        ),
                    }
                )
                global_i += 1

    return pd.DataFrame(rows)


scenario_table = build_scenario_table(CFG, ACTIVE_COUNTS)
scenario_table.head()

,sample_id,split,event_type,scenario_name,seed,event
0,0,train,fault,fault_00000,2026,"{'type': 'fault', 'bus': 1, 'tf': 1.0, 'tc': 1..."
1,1,train,fault,fault_00001,2027,"{'type': 'fault', 'bus': 3, 'tf': 1.0, 'tc': 1..."
2,2,train,fault,fault_00002,2028,"{'type': 'fault', 'bus': 5, 'tf': 1.0, 'tc': 1..."
3,3,train,fault,fault_00003,2029,"{'type': 'fault', 'bus': 4, 'tf': 1.0, 'tc': 1..."
4,4,train,fault,fault_00004,2030,"{'type': 'fault', 'bus': 1, 'tf': 1.0, 'tc': 1..."


## 6. Apply events and run ANDES

In [6]:
def _model_has_param_or_service(ss, model_name, src):
    return hasattr(ss, model_name) and hasattr(getattr(ss, model_name), src)


def add_fault(ss, event):
    payload = {
        "bus": event["bus"],
        "tf": event["tf"],
        "tc": event["tc"],
        "rf": event.get("rf", 0.0),
        "xf": event.get("xf", 0.01),
    }
    try:
        ss.add("Fault", payload)
    except Exception:
        ss.add("Fault", {"bus": event["bus"], "tf": event["tf"], "tc": event["tc"]})


def add_line_trip(ss, event):
    for line in event["lines"]:
        ss.add("Toggle", {"model": "Line", "dev": line, "t": event["trip_t"]})
        if event.get("reclose_t") is not None:
            ss.add("Toggle", {"model": "Line", "dev": line, "t": event["reclose_t"]})


def add_generator_shedding(ss, event):
    for gen in event["generators"]:
        ss.add(
            "Toggle", {"model": gen["model"], "dev": gen["dev"], "t": event["shed_t"]}
        )


def add_load_change(ss, event):
    for change in event["changes"]:
        model = change["model"]
        dev = change["dev"]
        method = change["method"]
        amount = change["amount"]
        t = event["change_t"]

        for src in [change["active_src"], change["reactive_src"]]:
            if not _model_has_param_or_service(ss, model, src):
                raise AttributeError(
                    f"Cannot apply load change: {model}.{src} does not exist. "
                    "Inspect the PQ model and adjust active_src/reactive_src."
                )

            ss.add(
                "Alter",
                {
                    "model": model,
                    "dev": dev,
                    "src": src,
                    "t": t,
                    "method": method,
                    "amount": amount,
                },
            )


def apply_event(ss, event):
    if event["type"] == "fault":
        add_fault(ss, event)
    elif event["type"] == "line_trip":
        add_line_trip(ss, event)
    elif event["type"] == "generator_shedding":
        add_generator_shedding(ss, event)
    elif event["type"] == "load_change":
        add_load_change(ss, event)
    elif event["type"] == "none":
        return
    else:
        raise ValueError(f"Unsupported event type: {event['type']}")


def validate_specs(ss, spec_dict, group_name):
    for label, (model_name, var_name) in spec_dict.items():
        if not hasattr(ss, model_name):
            raise AttributeError(f"[{group_name}] Missing model: {model_name}")
        if not hasattr(getattr(ss, model_name), var_name):
            raise AttributeError(
                f"[{group_name}] Missing variable: {model_name}.{var_name}"
            )


def disable_builtin_kundur_toggle(ss, cfg):
    if "kundur_full.xlsx" in str(cfg["case_file"]) and hasattr(ss, "Toggle"):
        try:
            ss.Toggle.alter("u", 1, 0)
        except Exception:
            pass


def apply_tds_options(ss, cfg, scenario):
    """
    Apply default and event-specific TDS options.

    Default settings apply to all events.
    Event-specific overrides are then applied based on scenario["event_type"].
    """
    # Core simulation settings.
    ss.TDS.config.tf = cfg["sim_tf"]
    ss.TDS.config.tstep = cfg["sim_tstep"]
    ss.TDS.config.fixt = 1

    tds_options = cfg.get("tds_options", {})

    # Apply default TDS options.
    for key, value in tds_options.get("default", {}).items():
        setattr(ss.TDS.config, key, value)

    # Apply event-specific overrides.
    event_type = scenario["event_type"]
    overrides = tds_options.get("event_overrides", {}).get(event_type, {})

    for key, value in overrides.items():
        setattr(ss.TDS.config, key, value)


def get_run_output_dir(cfg, scenario):
    return (
        Path(cfg["output_dir"])
        / scenario["split"]
        / scenario["event_type"]
        / f"sample_{int(scenario['sample_id']):06d}"
    )


def prepare_system_for_scenario(cfg, scenario):
    experiment_dir = Path(cfg["output_dir"])
    experiment_dir.mkdir(parents=True, exist_ok=True)

    ss = andes.load(
        cfg["case_file"],
        setup=False,
        output_path=str(experiment_dir),
        output="ieee14_full_pilot",
    )

    disable_builtin_kundur_toggle(ss, cfg)
    apply_event(ss, scenario["event"])

    ss.setup()

    validate_specs(ss, cfg["truth_vars"], "truth_vars")
    validate_specs(ss, cfg["meas_vars"], "meas_vars")

    pf_ok = ss.PFlow.run()
    if pf_ok is False:
        raise RuntimeError("Power flow did not converge.")

    apply_tds_options(ss, cfg, scenario)

    tds_ok = ss.TDS.run()
    if tds_ok is False:
        raise RuntimeError("TDS did not finish successfully.")

    return ss

## 7. Dataset extraction and saving

`truth.csv`, `meas_clean.csv`, and `meas_noisy.csv` use the same sampling grid.

In [7]:
def add_noise(df, noise_std, rng):
    noisy = df.copy()
    meta_rows = []

    for prefix, sigma in noise_std.items():
        cols = [c for c in noisy.columns if c.startswith(f"{prefix}__")]
        if not cols:
            continue

        if sigma > 0:
            noise = rng.normal(loc=0.0, scale=sigma, size=(len(noisy), len(cols)))
            noisy.loc[:, cols] = noisy[cols].to_numpy() + noise

        meta_rows.append(
            {"channel_prefix": prefix, "sigma": float(sigma), "n_cols": len(cols)}
        )

    return noisy, pd.DataFrame(meta_rows)


def make_dataset_from_run(ss, cfg, rng):
    truth_df = resample_df(extract_frame(ss, cfg["truth_vars"]), cfg["sample_tstep"])
    meas_clean_df = resample_df(
        extract_frame(ss, cfg["meas_vars"]), cfg["sample_tstep"]
    )

    common_index = truth_df.index.intersection(meas_clean_df.index)
    truth_df = truth_df.loc[common_index]
    meas_clean_df = meas_clean_df.loc[common_index]

    meas_noisy_df, noise_meta = add_noise(meas_clean_df, cfg["noise_std"], rng)

    return truth_df, meas_clean_df, meas_noisy_df, noise_meta


def save_run(cfg, scenario, truth_df, meas_clean_df, meas_noisy_df, noise_meta):
    run_dir = get_run_output_dir(cfg, scenario)
    run_dir.mkdir(parents=True, exist_ok=True)

    paths = {
        "config": run_dir / "config.json",
        "truth": run_dir / "truth.csv",
        "meas_clean": run_dir / "meas_clean.csv",
        "meas_noisy": run_dir / "meas_noisy.csv",
        "noise_meta": run_dir / "noise_meta.csv",
    }

    truth_df.to_csv(paths["truth"])
    meas_clean_df.to_csv(paths["meas_clean"])
    meas_noisy_df.to_csv(paths["meas_noisy"])
    noise_meta.to_csv(paths["noise_meta"], index=False)

    cfg_to_save = deepcopy(cfg)
    cfg_to_save["case_file"] = str(cfg_to_save["case_file"])
    cfg_to_save["output_dir"] = str(cfg_to_save["output_dir"])

    saved_config = {
        "global_config": cfg_to_save,
        "scenario": scenario,
        "interpretation": {
            "sample_definition": "one sample = one ANDES event trajectory",
            "input_file": "meas_noisy.csv",
            "target_file": "truth.csv",
            "clean_measurement_file": "meas_clean.csv",
        },
    }

    with open(paths["config"], "w") as f:
        json.dump(saved_config, f, indent=2)

    return {k: str(v) for k, v in paths.items()}

## 8. Smoke test one scenario

In [8]:
one_scenario = scenario_table.iloc[0].to_dict()
one_scenario["event"] = deepcopy(one_scenario["event"])

print("Scenario:")
print(json.dumps(one_scenario, indent=2))

rng = np.random.default_rng(one_scenario["seed"])
ss = prepare_system_for_scenario(CFG, one_scenario)

truth_df, meas_clean_df, meas_noisy_df, noise_meta = make_dataset_from_run(ss, CFG, rng)
paths = save_run(CFG, one_scenario, truth_df, meas_clean_df, meas_noisy_df, noise_meta)

print("\nSaved files:")
for k, v in paths.items():
    print(f"  {k}: {v}")

print("\nShapes:")
print("  truth:", truth_df.shape)
print("  meas_clean:", meas_clean_df.shape)
print("  meas_noisy:", meas_noisy_df.shape)

noise_meta

Scenario:
{
  "sample_id": 0,
  "split": "train",
  "event_type": "fault",
  "scenario_name": "fault_00000",
  "seed": 2026,
  "event": {
    "type": "fault",
    "bus": 1,
    "tf": 1.0,
    "tc": 1.0625254369572805,
    "rf": 0.009345368022869702,
    "xf": 0.03593503689756337
  }
}


  0%|                                                    | 0/100 [00:00<?, ?%/s]

<Fault Fault_1>: Applying fault on Bus (idx=1) at t=1.0 sec.
<Fault Fault_1>: Clearing fault on Bus (idx=1) at t=1.0625254369572805 sec.

Saved files:
  config: output/ieee14_full_pilot_v2/train/fault/sample_000000/config.json
  truth: output/ieee14_full_pilot_v2/train/fault/sample_000000/truth.csv
  meas_clean: output/ieee14_full_pilot_v2/train/fault/sample_000000/meas_clean.csv
  meas_noisy: output/ieee14_full_pilot_v2/train/fault/sample_000000/meas_noisy.csv
  noise_meta: output/ieee14_full_pilot_v2/train/fault/sample_000000/noise_meta.csv

Shapes:
  truth: (601, 20)
  meas_clean: (601, 28)
  meas_noisy: (601, 28)


,channel_prefix,sigma,n_cols
0,bus_v,0.002,14
1,bus_a,0.002,14


## 9. Inspect the smoke-test output

In [9]:
truth_df.head()

,gen_delta__GENROU_1,gen_delta__GENROU_2,gen_delta__GENROU_3,gen_delta__GENROU_4,gen_delta__GENROU_5,gen_omega__GENROU_1,gen_omega__GENROU_2,gen_omega__GENROU_3,gen_omega__GENROU_4,gen_omega__GENROU_5,gen_e1q__GENROU_1,gen_e1q__GENROU_2,gen_e1q__GENROU_3,gen_e1q__GENROU_4,gen_e1q__GENROU_5,gen_e1d__GENROU_1,gen_e1d__GENROU_2,gen_e1d__GENROU_3,gen_e1d__GENROU_4,gen_e1d__GENROU_5
t,,,,,,,,,,,,,,,,,,,,
0.000000,1.080456,0.340839,0.411893,0.204936,0.407489,1.0,1.0,1.0,1.0,1.0,0.844196,1.209498,1.0738,1.149208,1.059237,0.462687,0.170398,0.224235,0.151153,0.211037
0.016667,1.080456,0.340839,0.411893,0.204936,0.407489,1.0,1.0,1.0,1.0,1.0,0.844196,1.209498,1.0738,1.149208,1.059237,0.462687,0.170398,0.224235,0.151153,0.211037
0.033333,1.080456,0.340839,0.411893,0.204936,0.407489,1.0,1.0,1.0,1.0,1.0,0.844196,1.209498,1.0738,1.149208,1.059237,0.462687,0.170398,0.224235,0.151153,0.211037
0.050000,1.080456,0.340839,0.411893,0.204936,0.407489,1.0,1.0,1.0,1.0,1.0,0.844196,1.209498,1.0738,1.149208,1.059237,0.462687,0.170398,0.224235,0.151153,0.211037
0.066667,1.080456,0.340839,0.411893,0.204936,0.407489,1.0,1.0,1.0,1.0,1.0,0.844196,1.209498,1.0738,1.149208,1.059237,0.462687,0.170398,0.224235,0.151153,0.211037


In [10]:
meas_noisy_df.head()

,bus_v__1,bus_v__2,bus_v__3,bus_v__4,bus_v__5,bus_v__6,bus_v__7,bus_v__8,bus_v__9,bus_v__10,bus_v__11,bus_v__12,bus_v__13,bus_v__14,bus_a__1,bus_a__2,bus_a__3,bus_a__4,bus_a__5,bus_a__6,bus_a__7,bus_a__8,bus_a__9,bus_a__10,bus_a__11,bus_a__12,bus_a__13,bus_a__14
t,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0.000000,1.028414,1.030481,1.006207,1.014195,1.018532,1.029416,1.021848,1.030608,1.021233,1.015090,1.020555,1.018436,1.014322,1.016169,0.000330,-0.033483,-0.060663,-0.079348,-0.070781,-0.111318,-0.084880,-0.026646,-0.129493,-0.133547,-0.124161,-0.131644,-0.136592,-0.165665
0.016667,1.030322,1.028772,1.009192,1.012500,1.016995,1.027251,1.021517,1.031313,1.021304,1.015245,1.020399,1.021056,1.013024,1.019037,0.001455,-0.033039,-0.061744,-0.077438,-0.062810,-0.113671,-0.087319,-0.026710,-0.121048,-0.131570,-0.125225,-0.130827,-0.137106,-0.163993
0.033333,1.027540,1.030350,1.007661,1.014106,1.018923,1.032275,1.020700,1.031369,1.020731,1.014627,1.020128,1.019160,1.014859,1.015084,-0.001939,-0.030520,-0.062215,-0.075103,-0.064541,-0.111903,-0.086864,-0.026449,-0.127430,-0.128973,-0.124367,-0.130636,-0.135862,-0.165123
0.050000,1.028348,1.032889,1.011188,1.012843,1.021622,1.028368,1.027590,1.036302,1.025006,1.017196,1.017787,1.019396,1.013565,1.016297,0.004007,-0.027586,-0.062177,-0.076376,-0.070310,-0.111105,-0.083308,-0.029934,-0.128994,-0.132218,-0.123872,-0.132238,-0.133558,-0.164726
0.066667,1.029419,1.030568,1.012576,1.010292,1.015285,1.027994,1.020535,1.027138,1.019943,1.018129,1.017929,1.017921,1.012016,1.016679,-0.002868,-0.032668,-0.062441,-0.077600,-0.065273,-0.111460,-0.081732,-0.031255,-0.121080,-0.128118,-0.123978,-0.132368,-0.133238,-0.162542


In [11]:
truth_df.filter(like="gen_omega__").plot(
    figsize=(10, 4), grid=True, title="Truth: generator omega"
)
plt.xlabel("Time [s]")
plt.ylabel("omega [pu]")
plt.show()

meas_noisy_df.filter(like="bus_v__").plot(
    figsize=(10, 4), grid=True, title="Noisy measurements: bus voltage magnitude"
)
plt.xlabel("Time [s]")
plt.ylabel("voltage [pu]")
plt.show()

## 10. Batch generation

Generates every row in `scenario_table` and writes `manifest.csv`.

In [12]:
def max_delta_spread_deg(ss):
    """
    Compute maximum GENROU rotor-angle spread in degrees.

    This is useful for identifying loss-of-synchronism trajectories,
    especially when TDS.criteria is disabled.
    """
    if not hasattr(ss, "GENROU") or not hasattr(ss.GENROU, "delta"):
        return None

    try:
        delta_df = get_var_timeseries(ss, "GENROU", "delta")
        spread_rad = delta_df.max(axis=1) - delta_df.min(axis=1)
        return float(np.rad2deg(spread_rad.max()))
    except Exception:
        return None


def run_one_scenario(cfg, scenario):
    scenario = deepcopy(scenario)
    run_dir = get_run_output_dir(cfg, scenario)

    row = {
        "sample_id": scenario["sample_id"],
        "split": scenario["split"],
        "event_type": scenario["event_type"],
        "scenario_name": scenario["scenario_name"],
        "seed": scenario["seed"],
        "run_dir": str(run_dir),
        "status": "pending",
        "error": "",
        "n_time": None,
        "n_measurements": None,
        "n_states": None,
        "tds_criteria": None,
        "tds_tstep": None,
        "tds_tol": None,
        "tds_max_iter": None,
        "max_delta_spread_deg": None,
    }

    try:
        rng = np.random.default_rng(scenario["seed"])

        ss = prepare_system_for_scenario(cfg, scenario)

        truth_df, meas_clean_df, meas_noisy_df, noise_meta = make_dataset_from_run(
            ss, cfg, rng
        )
        save_run(cfg, scenario, truth_df, meas_clean_df, meas_noisy_df, noise_meta)

        row.update(
            {
                "status": "success",
                "n_time": len(meas_noisy_df),
                "n_measurements": meas_noisy_df.shape[1],
                "n_states": truth_df.shape[1],
                "tds_criteria": int(ss.TDS.config.criteria),
                "tds_tstep": float(ss.TDS.config.tstep),
                "tds_tol": float(ss.TDS.config.tol),
                "tds_max_iter": int(ss.TDS.config.max_iter),
                "max_delta_spread_deg": max_delta_spread_deg(ss),
            }
        )

    except Exception as exc:
        row.update({"status": "failed", "error": repr(exc)})

        if not cfg["continue_on_error"]:
            raise

    return row


manifest_rows = []

for _, df_row in scenario_table.iterrows():
    scenario = df_row.to_dict()
    scenario["event"] = deepcopy(scenario["event"])

    result = run_one_scenario(CFG, scenario)
    manifest_rows.append(result)

    print(
        f"[{result['status']}] "
        f"{result['split']} / {result['event_type']} / {result['scenario_name']} "
        f"-> {result['run_dir']}"
    )

manifest_df = pd.DataFrame(manifest_rows)

manifest_path = Path(CFG["output_dir"]) / "manifest.csv"
manifest_df.to_csv(manifest_path, index=False)

print("\nManifest saved to:", manifest_path)
manifest_df

  0%|                                                    | 0/100 [00:00<?, ?%/s]

<Fault Fault_1>: Applying fault on Bus (idx=1) at t=1.0 sec.
<Fault Fault_1>: Clearing fault on Bus (idx=1) at t=1.0625254369572805 sec.
[success] train / fault / fault_00000 -> output/ieee14_full_pilot_v2/train/fault/sample_000000


  0%|                                                    | 0/100 [00:00<?, ?%/s]

<Fault Fault_1>: Applying fault on Bus (idx=3) at t=1.0 sec.
<Fault Fault_1>: Clearing fault on Bus (idx=3) at t=1.0505603822284038 sec.
[success] train / fault / fault_00001 -> output/ieee14_full_pilot_v2/train/fault/sample_000001


  0%|                                                    | 0/100 [00:00<?, ?%/s]

...

  0%|                                                    | 0/100 [00:00<?, ?%/s]

<Alter Alter_1>: set PQ.PQ_9.Ppf.v=0.0515806 at t=1. Previous value was 0.061.
<Alter Alter_2>: set PQ.PQ_9.Qpf.v=0.0135293 at t=1. Previous value was 0.016.
[success] test / load_change / load_change_00000 -> output/ieee14_full_pilot_v2/test/load_change/sample_000027


  0%|                                                    | 0/100 [00:00<?, ?%/s]

<Alter Alter_1>: set PQ.PQ_10.Ppf.v=0.111604 at t=1. Previous value was 0.135.
<Alter Alter_2>: set PQ.PQ_10.Qpf.v=0.0479482 at t=1. Previous value was 0.058.
<Alter Alter_3>: set PQ.PQ_5.Ppf.v=0.1209 at t=1. Previous value was 0.15.
<Alter Alter_4>: set PQ.PQ_5.Qpf.v=0.0604498 at t=1. Previous value was 0.075.
<Alter Alter_5>: set PQ.PQ_6.Ppf.v=0.326945 at t=1. Previous value was 0.295.
<Alter Alter_6>: set PQ.PQ_6.Qpf.v=0.183976 at t=1. Previous value was 0.166.
[success] test / load_change / load_change_00001 -> output/ieee14_full_pilot_v2/test/load_change/sample_000028

Manifest saved to: output/ieee14_full_pilot_v2/manifest.csv


,sample_id,split,event_type,scenario_name,seed,run_dir,status,error,n_time,n_measurements,n_states,tds_criteria,tds_tstep,tds_tol,tds_max_iter,max_delta_spread_deg
0,0,train,fault,fault_00000,2026,output/ieee14_full_pilot_v2/train/fault/sample...,success,,601,28,20,1,0.008333,0.00010,15,62.804260
1,1,train,fault,fault_00001,2027,output/ieee14_full_pilot_v2/train/fault/sample...,success,,601,28,20,1,0.008333,0.00010,15,55.033285
2,2,train,fault,fault_00002,2028,output/ieee14_full_pilot_v2/train/fault/sample...,success,,601,28,20,1,0.008333,0.00010,15,56.730782
3,3,train,fault,fault_00003,2029,output/ieee14_full_pilot_v2/train/fault/sample...,success,,601,28,20,1,0.008333,0.00010,15,59.939993
4,4,train,fault,fault_00004,2030,output/ieee14_full_pilot_v2/train/fault/sample...,success,,601,28,20,1,0.008333,0.00010,15,65.033159
5,5,train,line_trip,line_trip_00000,2031,output/ieee14_full_pilot_v2/train/line_trip/sa...,success,,601,28,20,1,0.008333,0.00010,15,50.163589
6,6,train,line_trip,line_trip_00001,2032,output/ieee14_full_pilot_v2/train/line_trip/sa...,success,,601,28,20,1,0.008333,0.00010,15,53.660066
7,7,train,line_trip,line_trip_00002,2033,output/ieee14_full_pilot_v2/train/line_trip/sa...,success,,601,28,20,1,0.008333,0.00010,15,55.340083
8,8,train,line_trip,line_trip_00003,2034,output/ieee14_full_pilot_v2/train/line_trip/sa...,success,,601,28,20,1,0.008333,0.00010,15,53.902459
9,9,train,line_trip,line_trip_00004,2035,output/ieee14_full_pilot_v2/train/line_trip/sa...,success,,601,28,20,1,0.008333,0.00010,15,51.957162


The printed batch log has one line per attempted scenario.

- `[success]` means `truth.csv`, `meas_clean.csv`, `meas_noisy.csv`, and `noise_meta.csv` were written for that sample.
- `[failed]` means the exception was captured in `manifest_df["error"]`; the notebook keeps going because `CFG["continue_on_error"]` is enabled.
- The final displayed `manifest_df` is the complete run ledger and is the input to the summary checks below.

## 11. Dataset summary and failures

Use the manifest as a batch health check. Start with the aggregate status table, then inspect failed runs first. A clean pilot batch should have `failed = 0` and `success_rate = 1.0` for every split/event group.

In [13]:
from IPython.display import Markdown, display

required_cols = {
    "sample_id",
    "split",
    "event_type",
    "scenario_name",
    "status",
    "run_dir",
    "error",
}
missing_cols = sorted(required_cols - set(manifest_df.columns))
if missing_cols:
    raise ValueError(f"manifest_df is missing required columns: {missing_cols}")

status_order = ["success", "failed", "pending"]
status_dtype = pd.CategoricalDtype(categories=status_order, ordered=True)

summary_df = manifest_df.copy()
summary_df["status"] = summary_df["status"].astype(status_dtype)
summary_df["is_success"] = summary_df["status"].eq("success")

run_outcome = pd.DataFrame(
    {
        "total_runs": [len(summary_df)],
        "success": [int(summary_df["is_success"].sum())],
        "failed": [int(summary_df["status"].eq("failed").sum())],
        "pending": [int(summary_df["status"].eq("pending").sum())],
    }
)
run_outcome["success_rate"] = run_outcome["success"] / run_outcome[
    "total_runs"
].replace(0, np.nan)

status_by_group = (
    summary_df.groupby(["split", "event_type", "status"], observed=False)
    .size()
    .unstack("status", fill_value=0)
    .reindex(columns=status_order, fill_value=0)
    .reset_index()
)
status_by_group["total"] = status_by_group[status_order].sum(axis=1)
status_by_group["success_rate"] = status_by_group["success"] / status_by_group[
    "total"
].replace(0, np.nan)
status_by_group = status_by_group.sort_values(["split", "event_type"]).reset_index(
    drop=True
)

print("Overall run outcome")
display(run_outcome.style.format({"success_rate": "{:.1%}"}))

print("Status by split and event type")
display(status_by_group.style.format({"success_rate": "{:.1%}"}))

Overall run outcome


,total_runs,success,failed,pending,success_rate
0,29,29,0,0,100.0%


Status by split and event type


status,split,event_type,success,failed,pending,total,success_rate
0,test,fault,2,0,0,2,100.0%
1,test,generator_shedding,10,0,0,10,100.0%
2,test,line_trip,2,0,0,2,100.0%
3,test,load_change,2,0,0,2,100.0%
4,train,fault,5,0,0,5,100.0%
5,train,generator_shedding,0,0,0,0,nan%
6,train,line_trip,5,0,0,5,100.0%
7,train,load_change,3,0,0,3,100.0%


The next table is the failure triage view. If it is empty, every attempted scenario completed and wrote its sample files. If it is non-empty, sort by `event_type` and read `error` to identify the event family or ANDES setting that needs adjustment.

In [14]:
failed_runs = (
    summary_df.loc[
        ~summary_df["is_success"],
        [
            "sample_id",
            "split",
            "event_type",
            "scenario_name",
            "run_dir",
            "error",
        ],
    ]
    .sort_values(["split", "event_type", "sample_id"])
    .reset_index(drop=True)
)

if failed_runs.empty:
    display(Markdown("**No failed runs recorded.**"))
else:
    failure_counts = (
        failed_runs.groupby(["split", "event_type"])
        .size()
        .reset_index(name="failed")
        .sort_values(["split", "event_type"])
    )
    display(Markdown("**Failed-run counts**"))
    display(failure_counts)

    display(Markdown("**Failed-run details**"))
    display(failed_runs)

**No failed runs recorded.**

The success preview confirms which samples are usable for training or evaluation. Keep the solver columns beside each successful run so severe trajectories can be reviewed before modeling.

In [15]:
success_columns = [
    "sample_id",
    "split",
    "event_type",
    "scenario_name",
    "run_dir",
    "n_time",
    "n_measurements",
    "n_states",
    "tds_criteria",
    "tds_tstep",
    "tds_tol",
    "tds_max_iter",
    "max_delta_spread_deg",
]

available_success_columns = [c for c in success_columns if c in summary_df.columns]

successful_runs = (
    summary_df.loc[summary_df["is_success"], available_success_columns]
    .sort_values(["split", "event_type", "sample_id"])
    .reset_index(drop=True)
)

print(f"Successful runs: {len(successful_runs)}")
display(successful_runs)

Successful runs: 29


,sample_id,split,event_type,scenario_name,run_dir,n_time,n_measurements,n_states,tds_criteria,tds_tstep,tds_tol,tds_max_iter,max_delta_spread_deg
0,23,test,fault,fault_00000,output/ieee14_full_pilot_v2/test/fault/sample_...,601,28,20,1,0.008333,0.00010,15,64.698820
1,24,test,fault,fault_00001,output/ieee14_full_pilot_v2/test/fault/sample_...,601,28,20,1,0.008333,0.00010,15,58.645482
2,13,test,generator_shedding,generator_shedding_00000,output/ieee14_full_pilot_v2/test/generator_she...,601,28,20,0,0.004167,0.00001,30,824.051804
3,14,test,generator_shedding,generator_shedding_00001,output/ieee14_full_pilot_v2/test/generator_she...,601,28,20,0,0.004167,0.00001,30,515.219613
4,15,test,generator_shedding,generator_shedding_00002,output/ieee14_full_pilot_v2/test/generator_she...,601,28,20,0,0.004167,0.00001,30,913.232087
5,16,test,generator_shedding,generator_shedding_00003,output/ieee14_full_pilot_v2/test/generator_she...,601,28,20,0,0.004167,0.00001,30,515.219613
6,17,test,generator_shedding,generator_shedding_00004,output/ieee14_full_pilot_v2/test/generator_she...,601,28,20,0,0.004167,0.00001,30,913.232087
7,18,test,generator_shedding,generator_shedding_00005,output/ieee14_full_pilot_v2/test/generator_she...,601,28,20,0,0.004167,0.00001,30,913.232087
8,19,test,generator_shedding,generator_shedding_00006,output/ieee14_full_pilot_v2/test/generator_she...,601,28,20,0,0.004167,0.00001,30,2116.051827
9,20,test,generator_shedding,generator_shedding_00007,output/ieee14_full_pilot_v2/test/generator_she...,601,28,20,0,0.004167,0.00001,30,843.187140


In [16]:
summary = (
    manifest_df.groupby(["split", "event_type", "status"])
    .size()
    .reset_index(name="count")
)

summary

,split,event_type,status,count
0,test,fault,success,2
1,test,generator_shedding,success,10
2,test,line_trip,success,2
3,test,load_change,success,2
4,train,fault,success,5
5,train,line_trip,success,5
6,train,load_change,success,3


In [17]:
failed = manifest_df[manifest_df["status"] != "success"]
failed[["sample_id", "split", "event_type", "scenario_name", "error"]]

,sample_id,split,event_type,scenario_name,error


In [18]:
success_rate = (
    manifest_df.groupby(["split", "event_type"])["status"]
    .apply(lambda s: (s == "success").mean())
    .reset_index(name="success_rate")
)

success_rate

,split,event_type,success_rate
0,test,fault,1.0
1,test,generator_shedding,1.0
2,test,line_trip,1.0
3,test,load_change,1.0
4,train,fault,1.0
5,train,line_trip,1.0
6,train,load_change,1.0


In [19]:
manifest_df[
    [
        "sample_id",
        "split",
        "event_type",
        "scenario_name",
        "status",
        "tds_criteria",
        "tds_tstep",
        "tds_tol",
        "tds_max_iter",
        "max_delta_spread_deg",
        "error",
    ]
]

,sample_id,split,event_type,scenario_name,status,tds_criteria,tds_tstep,tds_tol,tds_max_iter,max_delta_spread_deg,error
0,0,train,fault,fault_00000,success,1,0.008333,0.00010,15,62.804260,
1,1,train,fault,fault_00001,success,1,0.008333,0.00010,15,55.033285,
2,2,train,fault,fault_00002,success,1,0.008333,0.00010,15,56.730782,
3,3,train,fault,fault_00003,success,1,0.008333,0.00010,15,59.939993,
4,4,train,fault,fault_00004,success,1,0.008333,0.00010,15,65.033159,
5,5,train,line_trip,line_trip_00000,success,1,0.008333,0.00010,15,50.163589,
6,6,train,line_trip,line_trip_00001,success,1,0.008333,0.00010,15,53.660066,
7,7,train,line_trip,line_trip_00002,success,1,0.008333,0.00010,15,55.340083,
8,8,train,line_trip,line_trip_00003,success,1,0.008333,0.00010,15,53.902459,
9,9,train,line_trip,line_trip_00004,success,1,0.008333,0.00010,15,51.957162,


## 12. Sample output-file EDA

Each successful sample directory contains the same trajectory saved in task-specific files.

- `meas_noisy.csv`: **main DSE input** `X`. PMU-like measurement channels after synthetic noise is added.
- `truth.csv`: **main DSE target** `Y`. Generator dynamic states selected by `CFG["truth_vars"]`.
- `meas_clean.csv`: noise-free measurement channels selected by `CFG["meas_vars"]`. Use for sanity checks, debugging, or oracle/noise-free ablations; not for the main noisy-measurement DSE benchmark.
- `noise_meta.csv`: noise configuration by channel prefix. Use as metadata/provenance, not as a time-series input.

Column convention for the three time-series files: the index is `t` in seconds, and columns are named `<channel_prefix>__<device_id>`. The prefix maps back to `CFG["meas_vars"]` or `CFG["truth_vars"]`.


In [21]:
from pathlib import Path

from IPython.display import Markdown, display

# Reuse the sample selected above. If this cell is run by itself after the batch,
# rebuild run_dir from the first successful manifest row.
if "run_dir" not in globals():
    success_rows = manifest_df[manifest_df["status"] == "success"]
    if len(success_rows) == 0:
        raise RuntimeError("No successful samples available for output-file EDA.")
    row = success_rows.iloc[0]
    run_dir = Path(row["run_dir"])

sample_file_roles = {
    "meas_noisy.csv": "DSE input X: noisy PMU-like measurements used for training/evaluation.",
    "truth.csv": "DSE target Y: ground-truth generator states to estimate.",
    "meas_clean.csv": "Noise-free measurements for checks and ablations; not the main benchmark input.",
    "noise_meta.csv": "Noise metadata by channel prefix; provenance, not a time series.",
}

sample_paths = {name: run_dir / name for name in sample_file_roles}
missing_files = [name for name, path in sample_paths.items() if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        f"Missing expected sample files in {run_dir}: {missing_files}"
    )

sample_tables = {
    "meas_noisy.csv": pd.read_csv(sample_paths["meas_noisy.csv"], index_col=0),
    "truth.csv": pd.read_csv(sample_paths["truth.csv"], index_col=0),
    "meas_clean.csv": pd.read_csv(sample_paths["meas_clean.csv"], index_col=0),
    "noise_meta.csv": pd.read_csv(sample_paths["noise_meta.csv"]),
}

role_df = pd.DataFrame(
    [
        {"file": name, "role": role, "path": str(sample_paths[name])}
        for name, role in sample_file_roles.items()
    ]
)

display(Markdown(f"**Sample run directory:** `{run_dir}`"))
display(Markdown("**File roles**"))
display(role_df)


def profile_timeseries_file(file_name, df):
    t = pd.to_numeric(pd.Index(df.index), errors="coerce").to_numpy(dtype=float)
    dt = np.diff(t)
    return {
        "file": file_name,
        "rows": len(df),
        "columns": df.shape[1],
        "t_start_s": np.nanmin(t) if len(t) else np.nan,
        "t_end_s": np.nanmax(t) if len(t) else np.nan,
        "median_dt_s": np.nanmedian(dt) if len(dt) else np.nan,
        "missing_values": int(df.isna().sum().sum()),
    }


time_series_files = ["meas_noisy.csv", "truth.csv", "meas_clean.csv"]
profile_df = pd.DataFrame(
    [profile_timeseries_file(name, sample_tables[name]) for name in time_series_files]
)

display(Markdown("**Time-series shape and time grid**"))
display(profile_df)


def source_lookup_from_cfg(cfg):
    lookup = {}
    for section in ["meas_vars", "truth_vars"]:
        for prefix, (model_name, var_name) in cfg[section].items():
            lookup[prefix] = f"{model_name}.{var_name}"
    return lookup


def channel_catalog(file_name, df, lookup):
    rows = []
    parsed = []
    for col in df.columns:
        if "__" in col:
            prefix, device_id = col.split("__", 1)
        else:
            prefix, device_id = "(unparsed)", col
        parsed.append((prefix, device_id, col))

    for prefix in sorted({x[0] for x in parsed}):
        group = [x for x in parsed if x[0] == prefix]
        device_ids = [x[1] for x in group]
        rows.append(
            {
                "file": file_name,
                "channel_prefix": prefix,
                "andes_source": lookup.get(prefix, "not found in CFG"),
                "n_columns": len(group),
                "example_columns": ", ".join([x[2] for x in group[:4]]),
                "example_device_ids": ", ".join(map(str, device_ids[:4])),
            }
        )
    return rows


lookup = source_lookup_from_cfg(CFG)
channel_catalog_df = pd.DataFrame(
    [
        row
        for name in time_series_files
        for row in channel_catalog(name, sample_tables[name], lookup)
    ]
)

display(Markdown("**Column groups and source variables**"))
display(channel_catalog_df)

display(Markdown("**noise_meta.csv**"))
display(sample_tables["noise_meta.csv"])

# Sanity check: noisy measurements should equal clean measurements plus synthetic noise.
common_meas_cols = sample_tables["meas_clean.csv"].columns.intersection(
    sample_tables["meas_noisy.csv"].columns
)
noise_delta = (
    sample_tables["meas_noisy.csv"][common_meas_cols]
    - sample_tables["meas_clean.csv"][common_meas_cols]
)
configured_sigma = (
    sample_tables["noise_meta.csv"].set_index("channel_prefix")["sigma"]
    if {"channel_prefix", "sigma"}.issubset(sample_tables["noise_meta.csv"].columns)
    else pd.Series(dtype=float)
)

noise_rows = []
for prefix in sorted(
    {col.split("__", 1)[0] for col in common_meas_cols if "__" in col}
):
    cols = [col for col in common_meas_cols if col.startswith(f"{prefix}__")]
    values = noise_delta[cols].to_numpy(dtype=float).ravel()
    noise_rows.append(
        {
            "channel_prefix": prefix,
            "n_columns": len(cols),
            "configured_sigma": configured_sigma.get(prefix, np.nan),
            "empirical_noise_mean": float(np.nanmean(values)),
            "empirical_noise_std": float(np.nanstd(values, ddof=1))
            if len(values) > 1
            else np.nan,
            "max_abs_noise": float(np.nanmax(np.abs(values)))
            if len(values)
            else np.nan,
        }
    )

noise_check_df = pd.DataFrame(noise_rows)

display(Markdown("**Noisy-minus-clean measurement check**"))
display(noise_check_df)

display(Markdown("**Preview: DSE input X (`meas_noisy.csv`)**"))
display(sample_tables["meas_noisy.csv"].head())

display(Markdown("**Preview: DSE target Y (`truth.csv`)**"))
display(sample_tables["truth.csv"].head())

**Sample run directory:** `output/ieee14_full_pilot_v2/train/fault/sample_000000`

**File roles**

,file,role,path
0,meas_noisy.csv,DSE input X: noisy PMU-like measurements used ...,output/ieee14_full_pilot_v2/train/fault/sample...
1,truth.csv,DSE target Y: ground-truth generator states to...,output/ieee14_full_pilot_v2/train/fault/sample...
2,meas_clean.csv,Noise-free measurements for checks and ablatio...,output/ieee14_full_pilot_v2/train/fault/sample...
3,noise_meta.csv,"Noise metadata by channel prefix; provenance, ...",output/ieee14_full_pilot_v2/train/fault/sample...


**Time-series shape and time grid**

,file,rows,columns,t_start_s,t_end_s,median_dt_s,missing_values
0,meas_noisy.csv,601,28,0.0,10.0,0.016667,0
1,truth.csv,601,20,0.0,10.0,0.016667,0
2,meas_clean.csv,601,28,0.0,10.0,0.016667,0


...

,channel_prefix,n_columns,configured_sigma,empirical_noise_mean,empirical_noise_std,max_abs_noise
0,bus_a,14,0.002,-1.780215e-05,0.001989,0.007335
1,bus_v,14,0.002,3.481437e-07,0.002018,0.007975


**Preview: DSE input X (`meas_noisy.csv`)**

,bus_v__1,bus_v__2,bus_v__3,bus_v__4,bus_v__5,bus_v__6,bus_v__7,bus_v__8,bus_v__9,bus_v__10,bus_v__11,bus_v__12,bus_v__13,bus_v__14,bus_a__1,bus_a__2,bus_a__3,bus_a__4,bus_a__5,bus_a__6,bus_a__7,bus_a__8,bus_a__9,bus_a__10,bus_a__11,bus_a__12,bus_a__13,bus_a__14
t,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0.000000,1.028414,1.030481,1.006207,1.014195,1.018532,1.029416,1.021848,1.030608,1.021233,1.015090,1.020555,1.018436,1.014322,1.016169,0.000330,-0.033483,-0.060663,-0.079348,-0.070781,-0.111318,-0.084880,-0.026646,-0.129493,-0.133547,-0.124161,-0.131644,-0.136592,-0.165665
0.016667,1.030322,1.028772,1.009192,1.012500,1.016995,1.027251,1.021517,1.031313,1.021304,1.015245,1.020399,1.021056,1.013024,1.019037,0.001455,-0.033039,-0.061744,-0.077438,-0.062810,-0.113671,-0.087319,-0.026710,-0.121048,-0.131570,-0.125225,-0.130827,-0.137106,-0.163993
0.033333,1.027540,1.030350,1.007661,1.014106,1.018923,1.032275,1.020700,1.031369,1.020731,1.014627,1.020128,1.019160,1.014859,1.015084,-0.001939,-0.030520,-0.062215,-0.075103,-0.064541,-0.111903,-0.086864,-0.026449,-0.127430,-0.128973,-0.124367,-0.130636,-0.135862,-0.165123
0.050000,1.028348,1.032889,1.011188,1.012843,1.021622,1.028368,1.027590,1.036302,1.025006,1.017196,1.017787,1.019396,1.013565,1.016297,0.004007,-0.027586,-0.062177,-0.076376,-0.070310,-0.111105,-0.083308,-0.029934,-0.128994,-0.132218,-0.123872,-0.132238,-0.133558,-0.164726
0.066667,1.029419,1.030568,1.012576,1.010292,1.015285,1.027994,1.020535,1.027138,1.019943,1.018129,1.017929,1.017921,1.012016,1.016679,-0.002868,-0.032668,-0.062441,-0.077600,-0.065273,-0.111460,-0.081732,-0.031255,-0.121080,-0.128118,-0.123978,-0.132368,-0.133238,-0.162542


**Preview: DSE target Y (`truth.csv`)**

,gen_delta__GENROU_1,gen_delta__GENROU_2,gen_delta__GENROU_3,gen_delta__GENROU_4,gen_delta__GENROU_5,gen_omega__GENROU_1,gen_omega__GENROU_2,gen_omega__GENROU_3,gen_omega__GENROU_4,gen_omega__GENROU_5,gen_e1q__GENROU_1,gen_e1q__GENROU_2,gen_e1q__GENROU_3,gen_e1q__GENROU_4,gen_e1q__GENROU_5,gen_e1d__GENROU_1,gen_e1d__GENROU_2,gen_e1d__GENROU_3,gen_e1d__GENROU_4,gen_e1d__GENROU_5
t,,,,,,,,,,,,,,,,,,,,
0.000000,1.080456,0.340839,0.411893,0.204936,0.407489,1.0,1.0,1.0,1.0,1.0,0.844196,1.209498,1.0738,1.149208,1.059237,0.462687,0.170398,0.224235,0.151153,0.211037
0.016667,1.080456,0.340839,0.411893,0.204936,0.407489,1.0,1.0,1.0,1.0,1.0,0.844196,1.209498,1.0738,1.149208,1.059237,0.462687,0.170398,0.224235,0.151153,0.211037
0.033333,1.080456,0.340839,0.411893,0.204936,0.407489,1.0,1.0,1.0,1.0,1.0,0.844196,1.209498,1.0738,1.149208,1.059237,0.462687,0.170398,0.224235,0.151153,0.211037
0.050000,1.080456,0.340839,0.411893,0.204936,0.407489,1.0,1.0,1.0,1.0,1.0,0.844196,1.209498,1.0738,1.149208,1.059237,0.462687,0.170398,0.224235,0.151153,0.211037
0.066667,1.080456,0.340839,0.411893,0.204936,0.407489,1.0,1.0,1.0,1.0,1.0,0.844196,1.209498,1.0738,1.149208,1.059237,0.462687,0.170398,0.224235,0.151153,0.211037


For the standard supervised DSE dataset, load each successful sample as:

```python
X = pd.read_csv(run_dir / "meas_noisy.csv", index_col=0)
Y = pd.read_csv(run_dir / "truth.csv", index_col=0)
```

Keep `meas_clean.csv` out of the main model input unless the experiment is explicitly a noise-free/oracle ablation. Keep `noise_meta.csv` as metadata for documenting or conditioning on the noise model.

## 13. Reload one generated sample

This is the pattern a later training notebook should use.

In [20]:
success_rows = manifest_df[manifest_df["status"] == "success"]

if len(success_rows) == 0:
    raise RuntimeError("No successful samples available to reload.")

row = success_rows.iloc[0]
run_dir = Path(row["run_dir"])

X = pd.read_csv(run_dir / "meas_noisy.csv", index_col=0)
Y = pd.read_csv(run_dir / "truth.csv", index_col=0)

print("Run directory:", run_dir)
print("X shape:", X.shape)
print("Y shape:", Y.shape)

X.head(), Y.head()

Run directory: output/ieee14_full_pilot_v2/train/fault/sample_000000
X shape: (601, 28)
Y shape: (601, 20)


(          bus_v__1  bus_v__2  bus_v__3  bus_v__4  bus_v__5  bus_v__6  bus_v__7  bus_v__8  bus_v__9  bus_v__10  bus_v__11  bus_v__12  bus_v__13  bus_v__14  bus_a__1  bus_a__2  bus_a__3  bus_a__4  \
 t                                                                                                                                                                                                   
 0.000000  1.028414  1.030481  1.006207  1.014195  1.018532  1.029416  1.021848  1.030608  1.021233   1.015090   1.020555   1.018436   1.014322   1.016169  0.000330 -0.033483 -0.060663 -0.079348   
 0.016667  1.030322  1.028772  1.009192  1.012500  1.016995  1.027251  1.021517  1.031313  1.021304   1.015245   1.020399   1.021056   1.013024   1.019037  0.001455 -0.033039 -0.061744 -0.077438   
 0.033333  1.027540  1.030350  1.007661  1.014106  1.018923  1.032275  1.020700  1.031369  1.020731   1.014627   1.020128   1.019160   1.014859   1.015084 -0.001939 -0.030520 -0.062215 -0.075103   
 0.050000 

## 14. Scaling notes

To generate the paper-like 5000-trajectory dataset:

1. Run the smoke test.
2. Run `COUNTS_PILOT` and inspect `manifest.csv`.
3. Fix failed event types, especially `load_change`, because PQ-load internals may vary across cases.
4. Change:

```python
ACTIVE_COUNTS = COUNTS_PAPER_LIKE
```

5. Restart the kernel and run all cells.

If many fault scenarios fail, reduce severity by increasing the lower bound of `fault_xf`, shortening the fault duration range, or reducing simultaneous device trips.